## Strategie
- wirte a parser for the codebook, to extract qname, year, question, comment, note
- identify the three typse of variables: fixed (same qname across years), changing (qname has year suffix), and retroperspectiv (qname appears only once in 1988, and is ask retrospectivly for the R's age 0-18).
- how to identefy them? fixed variables asked only once? not only in 1979 but also later on, changing are asked more than once, retroperspectiv variables are asked only once in 1988, and have a question that asks for the R's age 0-18.
- open csv files, and for each variable, assign qname, year, question, comment, note from the codebook.
- create two datasets p (fixed) and pl (change over years),
- reshape pl to long format --> pd.melt , carufle with retroperspectiv variables -> need to change syear = 1988-R's age + childhood age which is asked, and qname = qname without year suffix + "_" + childhood age which is asked.
- merge p and pl for syear and id 1:1, and export to csv
- create two datasets, (1) pregnant women, (2) young adults

In [15]:
csv_path = "/home/theo/PycharmProjects/Bookoflife/Original/data/raw/data.csv"
cdb_path = "/home/theo/PycharmProjects/Bookoflife/Original/data/raw/data.cbd"


In [17]:
import re
import pandas as pd
from pathlib import Path

def parse_cdb(cdb_path):
    """Parse NLSY79 codebook (.cdb) - simple version"""
    codebook = {}
    text = Path(cdb_path).read_text()

    # Split by long dashes
    blocks = re.split(r"-{40,}", text)

    for block in blocks:
        # Match header: H00016.00    [H40-BPAR-1]    Survey Year: XRND
        m = re.match(r"\s*(\w+\.\d+)\s+\[([^\]]+)\]\s+Survey Year:\s*(\S+)", block)
        if not m:
            continue

        rnum = m.group(1).replace(".", "")  # H0001600
        qname = m.group(2)                   # H40-BPAR-1
        year = m.group(3)                    # XRND

        # Extract question (after "PRIMARY VARIABLE")
        question = ""
        qm = re.search(r"PRIMARY VARIABLE\s*\n\s*\n\s*(.+?)(?:\n\s*\n|$)", block, re.DOTALL)
        if qm:
            question = qm.group(1).strip().replace("\n", " ")

        codebook[rnum] = {
            "rnum": rnum,
            "qname": qname,
            "year": year,
            "question": question,
        }

    return codebook


In [20]:
cdb_path = "/home/theo/PycharmProjects/Bookoflife/Original/data/raw/data.cdb"
codebook = parse_cdb(cdb_path)
print(codebook)
# Speichere als CSV
#out_dir = Path("/home/theo/PycharmProjects/Bookoflife/Original/data/processed")
#out_dir.mkdir(parents=True, exist_ok=True)

cb_df = pd.DataFrame(codebook.values())
#cb_df.to_csv(out_dir / "codebook.csv", index=False)


{'H0001600': {'rnum': 'H0001600', 'qname': 'H40-BPAR-1', 'year': 'XRND', 'question': "R'S BIOLOGICAL FATHER LIVING?"}, 'H0002400': {'rnum': 'H0002400', 'qname': 'H40-BPAR-6', 'year': 'XRND', 'question': "R'S BIOLOGICAL MOTHER LIVING?"}, 'H0013600': {'rnum': 'H0013600', 'qname': 'H50BPAR-1', 'year': 'XRND', 'question': "R'S BIOLOGICAL FATHER LIVING?"}, 'H0014700': {'rnum': 'H0014700', 'qname': 'H50BPAR-6', 'year': 'XRND', 'question': "R'S BIOLOGICAL MOTHER LIVING?"}, 'H0046300': {'rnum': 'H0046300', 'qname': 'H60BPAR-1', 'year': 'XRND', 'question': "R'S BIOLOGICAL FATHER LIVING?"}, 'H0048900': {'rnum': 'H0048900', 'qname': 'H60BPAR-6', 'year': 'XRND', 'question': "R'S BIOLOGICAL MOTHER LIVING?"}, 'R0000100': {'rnum': 'R0000100', 'qname': 'CASEID', 'year': '1979', 'question': 'IDENTIFICATION CODE'}, 'R0000600': {'rnum': 'R0000600', 'qname': 'FAM-1B', 'year': '1979', 'question': 'AGE OF R'}, 'R0002000': {'rnum': 'R0002000', 'qname': 'FAM-8', 'year': '1979', 'question': 'RELATION TO R OF A

In [23]:
cb_df["year"] !=

,rnum,qname,year,question
0,H0001600,H40-BPAR-1,XRND,R'S BIOLOGICAL FATHER LIVING?
1,H0002400,H40-BPAR-6,XRND,R'S BIOLOGICAL MOTHER LIVING?
2,H0013600,H50BPAR-1,XRND,R'S BIOLOGICAL FATHER LIVING?
3,H0014700,H50BPAR-6,XRND,R'S BIOLOGICAL MOTHER LIVING?
4,H0046300,H60BPAR-1,XRND,R'S BIOLOGICAL FATHER LIVING?
...,...,...,...,...
460,T8018500,Q9-89.04,2018,FEMALE - USE MARIJUANA IN 12 MOS BEFORE PREGNA...
461,T8018600,Q9-91.01,2018,FEMALE - USE COCAINE IN 12 MOS BEFORE PREGNANC...
462,T8018700,Q9-91.02,2018,FEMALE - USE COCAINE IN 12 MOS BEFORE PREGNANC...
463,T8018800,Q9-91.03,2018,FEMALE - USE COCAINE IN 12 MOS BEFORE PREGNANC...


In [43]:
cb_df["qname"].value_counts()

qname
Q9-89.01    13
Q9-91.01    13
Q9-89.02    13
Q9-91.02    13
Q9-88.01    10
            ..
Q9-90.04     1
Q9-91.07     1
Q9-91.08     1
Q9-89.10     1
Q9-91.10     1
Name: count, Length: 321, dtype: int64

In [47]:
not_year = cb_df[~cb_df["year"].astype(str).str.fullmatch(r"\d{4}")]

        rnum        qname    year                                question
0   H0001600   H40-BPAR-1    XRND           R'S BIOLOGICAL FATHER LIVING?
1   H0002400   H40-BPAR-6    XRND           R'S BIOLOGICAL MOTHER LIVING?
2   H0013600    H50BPAR-1    XRND           R'S BIOLOGICAL FATHER LIVING?
3   H0014700    H50BPAR-6    XRND           R'S BIOLOGICAL MOTHER LIVING?
4   H0046300    H60BPAR-1    XRND           R'S BIOLOGICAL FATHER LIVING?
5   H0048900    H60BPAR-6    XRND           R'S BIOLOGICAL MOTHER LIVING?
17  R0214700  SAMPLE_RACE  78SCRN  R'S RACIAL/ETHNIC COHORT FROM SCREENER


In [ ]:
def reshape(csv_path, cdb_path, out_dir):
    """Simple reshape: parse cdb + read csv"""

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1. Parse codebook
    codebook = parse_cdb(cdb_path)
    print(f"✓ Codebook parsed: {len(codebook)} variables")

    # 2. Read CSV
    raw = pd.read_csv(csv_path)
    print(f"✓ CSV loaded: {raw.shape[0]} rows × {raw.shape[1]} columns")

    # 3. Save codebook as CSV for inspection
    cb_df = pd.DataFrame(codebook.values())
    cb_df.to_csv(out_dir / "codebook.csv", index=False)
    print(f" Codebook saved to: {out_dir / 'codebook.csv'}")

    return codebook, raw


In [ ]:
codebook, raw = reshape(path_csv, path_cbd, "/home/theo/PycharmProjects/Bookoflife/Original/data/processed")
